# 00 — Data creation

### How to read this notebook
Run cells top to bottom. Each Markdown cell states the question and expected result **before** the code. The printed and plotted numbers are measurements, not claims made in advance. The shared functions live in `src/experiment.py`. Open Jupyter from the project root so imports work.

## Target and features

We sample binary labels $Y\in\{0,1\}$ from a balanced population and independent $U\in\{-1,+1\}$. The finite sample is not forced to be exactly balanced. Set $T=2Y-1$. Then $X_s=T$ and $X_c=(U,UT)$. The complete scaled input is $X_\alpha=[T,\alpha U,\alpha UT]$.

The simple rule is $Y=\mathbf 1[X_s>0]$. The complex rule is $Y=\mathbf 1[X_{c1}X_{c2}>0]$. Both perfectly reveal the target, although the latter is not linearly separable.

In [1]:
from pathlib import Path
import os
import sys

# Resolve this experiment before imports and relative result paths.
ROOT = Path.cwd()
if not (ROOT / 'src' / 'experiment.py').is_file():
    ROOT = ROOT / 'experiment_1'
assert (ROOT / 'src' / 'experiment.py').is_file(), 'Start Jupyter in SimpBias or experiment_1.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.experiment import *
import torch
import numpy as np
import matplotlib.pyplot as plt
print('PyTorch:', torch.__version__)

PyTorch: 2.14.1+cpu


In [2]:
x,y=make_data(n=1000,seed=42,alpha=1.0)
print('Shapes:',x.shape,y.shape)
print('First five examples:\n',torch.cat([x,y],dim=1)[:5])
print('Simple rule correct:',bool(((x[:,0]>0).float()==y[:,0]).all()))
print('Complex rule correct:',bool(((x[:,1]*x[:,2]>0).float()==y[:,0]).all()))

Shapes: torch.Size([1000, 3]) torch.Size([1000, 1])
First five examples:
 tensor([[-1.,  1., -1.,  0.],
        [ 1.,  1.,  1.,  1.],
        [ 1.,  1.,  1.,  1.],
        [-1.,  1., -1.,  0.],
        [-1., -1.,  1.,  0.]])
Simple rule correct: True
Complex rule correct: True


## Check the four possible patterns

The dataset has only **four distinct prototypes**. This is a controlled toy setting, not a realistic generalization benchmark.

In [3]:
print('Unique [simple, complex 1, complex 2, label] rows:')
print(torch.unique(torch.cat([x,y],dim=1),dim=0))

Unique [simple, complex 1, complex 2, label] rows:
tensor([[-1., -1.,  1.,  0.],
        [-1.,  1., -1.,  0.],
        [ 1., -1., -1.,  1.],
        [ 1.,  1.,  1.,  1.]])


## Check scaling invariance

For any $\alpha>0$, scaling does not change the sign of $X_{c1}X_{c2}$, so the complex label rule is unchanged. We verify this directly.

In [4]:
for alpha in [0.1,0.5,1,2,5,10]:
    xx,yy=make_data(1000,42,alpha)
    correct=((xx[:,1]*xx[:,2]>0).float()==yy[:,0]).float().mean().item()
    print(f'alpha={alpha:>4}: complex-rule accuracy={correct:.3f}')

alpha= 0.1: complex-rule accuracy=1.000
alpha= 0.5: complex-rule accuracy=1.000
alpha=   1: complex-rule accuracy=1.000
alpha=   2: complex-rule accuracy=1.000
alpha=   5: complex-rule accuracy=1.000
alpha=  10: complex-rule accuracy=1.000
